In [13]:
# ============================================================
# STEP 1 - CELL 1
# SETUP
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow.parquet as pq

import os
import gc
import json
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/step1")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 1 - DATA PREPARATION")
print("=" * 70)

print("Input :", INPUT_DIR)
print("Output:", OUTPUT_DIR)

print("\n✅ Setup completed")

STEP 1 - DATA PREPARATION
Input : /kaggle/input
Output: /kaggle/working/step1

✅ Setup completed


In [14]:
# ============================================================
# STEP 1 - CELL 2
# DISCOVER INPUT FILES
# ============================================================

print("=" * 70)
print("DISCOVER INPUT FILES")
print("=" * 70)

all_files = sorted([
    p for p in INPUT_DIR.rglob("*")
    if p.is_file()
])

print(f"\nTotal files: {len(all_files):,}\n")

for i, path in enumerate(all_files, 1):

    try:
        size_mb = path.stat().st_size / (1024 ** 2)
    except:
        size_mb = 0

    print(
        f"{i:02d}. {path.name:<55} "
        f"{size_mb:>9.2f} MB"
    )

    print(f"    {path}")

DISCOVER INPUT FILES

Total files: 22

01. english_reviews.csv                                        516.70 MB
    /kaggle/input/datasets/trnquangtriu/1bdataset/english_reviews.csv
02. multilingual_reviews.csv                                    46.57 MB
    /kaggle/input/datasets/trnquangtriu/1bdataset/multilingual_reviews.csv
03. vietnamese_reviews.csv                                      18.23 MB
    /kaggle/input/datasets/trnquangtriu/1bdataset/vietnamese_reviews.csv
04. full-00000-of-00034.parquet                                303.06 MB
    /kaggle/input/datasets/trnquangtriu/1cdataset/full-00000-of-00034.parquet
05. full-00000-of-00010.parquet                                209.73 MB
    /kaggle/input/datasets/trnquangtriu/1dataset/full-00000-of-00010.parquet
06. full-00001-of-00010.parquet                                204.18 MB
    /kaggle/input/datasets/trnquangtriu/1dataset/full-00001-of-00010.parquet
07. full-00002-of-00010.parquet                                200.03 MB


In [15]:
# ============================================================
# STEP 1 - CELL 3
# CLASSIFY DATA SOURCES
# ============================================================

# ------------------------------------------------------------
# Amazon metadata: 10 shards
# ------------------------------------------------------------

amazon_metadata_files = sorted([
    p for p in all_files
    if p.suffix == ".parquet"
    and "of-00010" in p.name
])


# ------------------------------------------------------------
# Amazon reviews: currently 1 shard of 34
# ------------------------------------------------------------

amazon_review_files = sorted([
    p for p in all_files
    if p.suffix == ".parquet"
    and "of-00034" in p.name
])


# ------------------------------------------------------------
# Tiki products
# ------------------------------------------------------------

tiki_files = sorted([
    p for p in all_files
    if p.suffix == ".csv"
    and p.name.startswith("vietnamese_tiki_products_")
])


# ------------------------------------------------------------
# Old review datasets
# ------------------------------------------------------------

old_english_files = [
    p for p in all_files
    if p.name == "english_reviews.csv"
]

old_vietnamese_files = [
    p for p in all_files
    if p.name == "vietnamese_reviews.csv"
]

old_multilingual_files = [
    p for p in all_files
    if p.name == "multilingual_reviews.csv"
]


# ------------------------------------------------------------
# Shopee sentiment files
# ------------------------------------------------------------

sentiment_files = [
    p for p in all_files
    if p.name in ["train.csv", "val.csv"]
]


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 70)
print("DATA SOURCE SUMMARY")
print("=" * 70)

print(
    "Amazon metadata shards :",
    len(amazon_metadata_files)
)

print(
    "Amazon review shards   :",
    len(amazon_review_files)
)

print(
    "Tiki product files     :",
    len(tiki_files)
)

print(
    "Old English reviews    :",
    len(old_english_files)
)

print(
    "Old Vietnamese reviews :",
    len(old_vietnamese_files)
)

print(
    "Old multilingual data  :",
    len(old_multilingual_files)
)

print(
    "Sentiment files        :",
    len(sentiment_files)
)


assert len(amazon_metadata_files) == 10, \
    "❌ Expected 10 Amazon metadata shards"

assert len(amazon_review_files) >= 1, \
    "❌ Amazon review shard not found"

assert len(tiki_files) == 6, \
    "❌ Expected 6 Tiki CSV files"

print("\n✅ Required datasets detected")

DATA SOURCE SUMMARY
Amazon metadata shards : 10
Amazon review shards   : 1
Tiki product files     : 6
Old English reviews    : 1
Old Vietnamese reviews : 1
Old multilingual data  : 1
Sentiment files        : 2

✅ Required datasets detected


# AMAZON REVIEW SCHEMA


In [16]:
# ============================================================
# STEP 1 - CELL 4
# AMAZON REVIEW SCHEMA
# ============================================================

print("=" * 70)
print("AMAZON REVIEW SCHEMA")
print("=" * 70)

review_path = amazon_review_files[0]

print("\nFile:")
print(review_path)

schema = pq.read_schema(review_path)

print("\nColumns:")

for field in schema:
    print(
        f"{field.name:<25} "
        f"{field.type}"
    )


pf = pq.ParquetFile(review_path)

print("\nParquet information")
print("-" * 70)

print(
    "Rows       :",
    f"{pf.metadata.num_rows:,}"
)

print(
    "Columns    :",
    pf.metadata.num_columns
)

print(
    "Row groups :",
    pf.metadata.num_row_groups
)

AMAZON REVIEW SCHEMA

File:
/kaggle/input/datasets/trnquangtriu/1cdataset/full-00000-of-00034.parquet

Columns:
rating                    double
title                     string
text                      string
images                    list<element: struct<attachment_type: string, large_image_url: string, medium_image_url: string, small_image_url: string>>
asin                      string
parent_asin               string
user_id                   string
timestamp                 int64
helpful_vote              int64
verified_purchase         bool

Parquet information
----------------------------------------------------------------------
Rows       : 1,290,793
Columns    : 13
Row groups : 1291


# LOAD AMAZON REVIEW SHARD

In [17]:
# ============================================================
# STEP 1 - CELL 5
# LOAD AMAZON REVIEW SHARD
# ============================================================

required_review_cols = [
    "rating",
    "title",
    "text",
    "asin",
    "parent_asin",
    "user_id",
    "timestamp",
    "verified_purchase",
    "helpful_vote"
]

available_cols = pq.read_schema(
    amazon_review_files[0]
).names

review_cols = [
    c for c in required_review_cols
    if c in available_cols
]

print("Loading columns:")
print(review_cols)


amazon_reviews = pd.read_parquet(
    amazon_review_files[0],
    columns=review_cols
)


print("\n" + "=" * 70)
print("AMAZON REVIEW DATA")
print("=" * 70)

print(
    "Rows    :",
    f"{len(amazon_reviews):,}"
)

print(
    "Columns :",
    len(amazon_reviews.columns)
)

print(
    "Users   :",
    f"{amazon_reviews['user_id'].nunique():,}"
)

print(
    "ASIN    :",
    f"{amazon_reviews['asin'].nunique():,}"
)

print(
    "Products:",
    f"{amazon_reviews['parent_asin'].nunique():,}"
)

display(
    amazon_reviews.head()
)

Loading columns:
['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'verified_purchase', 'helpful_vote']

AMAZON REVIEW DATA
Rows    : 1,290,793
Columns : 9
Users   : 245,477
ASIN    : 371,944
Products: 315,544


,rating,title,text,asin,parent_asin,user_id,timestamp,verified_purchase,helpful_vote
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline so if you are sensitive/allergic to petroleum prod...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,True,0
1,1.0,Didn’t work at all lenses loose/broken.,"These didn’t work. Idk if they were damaged in shipping or what, but the lenses were loose or so...",B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,True,0
2,5.0,Excellent!,I love these. They even come with a carry case and several sizes of ear bud inserts. Thank heave...,B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,True,0
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for school that would allow me to carry my laptop as well ...,B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,True,18
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times because I love them so much and overuse them and find w...,B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,True,0


# Làm sạch Amazon Review IDs

In [18]:
# ============================================================
# STEP 1 - CELL 6
# CLEAN AMAZON REVIEW DATA
# ============================================================

amazon_reviews["parent_asin"] = (
    amazon_reviews["parent_asin"]
    .astype("string")
    .str.strip()
    .str.upper()
)

amazon_reviews["asin"] = (
    amazon_reviews["asin"]
    .astype("string")
    .str.strip()
    .str.upper()
)

amazon_reviews["user_id"] = (
    amazon_reviews["user_id"]
    .astype("string")
    .str.strip()
)

amazon_reviews["rating"] = pd.to_numeric(
    amazon_reviews["rating"],
    errors="coerce"
)

amazon_reviews["text"] = (
    amazon_reviews["text"]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# Valid rows
# ------------------------------------------------------------

valid_amazon_review = (
    amazon_reviews["parent_asin"].notna()
    &
    amazon_reviews["user_id"].notna()
    &
    amazon_reviews["rating"].between(1, 5)
)

print("=" * 70)
print("AMAZON REVIEW QUALITY")
print("=" * 70)

print(
    "Total rows :",
    f"{len(amazon_reviews):,}"
)

print(
    "Valid rows :",
    f"{valid_amazon_review.sum():,}"
)

print(
    "Missing user_id:",
    f"{amazon_reviews['user_id'].isna().sum():,}"
)

print(
    "Missing parent_asin:",
    f"{amazon_reviews['parent_asin'].isna().sum():,}"
)

print(
    "Missing text:",
    f"{amazon_reviews['text'].isna().sum():,}"
)


amazon_reviews = (
    amazon_reviews[
        valid_amazon_review
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "\nRows retained:",
    f"{len(amazon_reviews):,}"
)

AMAZON REVIEW QUALITY
Total rows : 1,290,793
Valid rows : 1,290,793
Missing user_id: 0
Missing parent_asin: 0
Missing text: 0

Rows retained: 1,290,793


# Load Amazon Metadata

In [19]:
# ============================================================
# STEP 1 - CELL 7
# LOAD AMAZON PRODUCT METADATA
# ============================================================

amazon_product_cols = [
    "parent_asin",
    "title",
    "main_category",
    "average_rating",
    "rating_number",
    "price",
    "store"
]

metadata_parts = []

for path in amazon_metadata_files:

    temp = pd.read_parquet(
        path,
        columns=amazon_product_cols
    )

    metadata_parts.append(temp)

    print(
        f"{path.name:<35}"
        f"{len(temp):>12,}"
    )


amazon_products = pd.concat(
    metadata_parts,
    ignore_index=True
)

del metadata_parts
gc.collect()


# ------------------------------------------------------------
# Normalize
# ------------------------------------------------------------

amazon_products["parent_asin"] = (
    amazon_products["parent_asin"]
    .astype("string")
    .str.strip()
    .str.upper()
)

amazon_products["title"] = (
    amazon_products["title"]
    .astype("string")
    .str.strip()
)

amazon_products["main_category"] = (
    amazon_products["main_category"]
    .astype("string")
    .str.strip()
)

for col in [
    "average_rating",
    "rating_number",
    "price"
]:
    amazon_products[col] = pd.to_numeric(
        amazon_products[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# Remove duplicate parent ASIN
# ------------------------------------------------------------

amazon_products = (
    amazon_products
    .sort_values(
        "rating_number",
        ascending=False
    )
    .drop_duplicates(
        "parent_asin",
        keep="first"
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("AMAZON METADATA")
print("=" * 70)

print(
    "Products:",
    f"{len(amazon_products):,}"
)

print(
    "Unique parent_asin:",
    f"{amazon_products['parent_asin'].nunique():,}"
)

full-00000-of-00010.parquet             161,002
full-00001-of-00010.parquet             161,002
full-00002-of-00010.parquet             161,001
full-00003-of-00010.parquet             161,001
full-00004-of-00010.parquet             161,001
full-00005-of-00010.parquet             161,001
full-00006-of-00010.parquet             161,001
full-00007-of-00010.parquet             161,001
full-00008-of-00010.parquet             161,001
full-00009-of-00010.parquet             161,001

AMAZON METADATA
Products: 1,610,006
Unique parent_asin: 1,610,006


# Tạo Amazon Candidate Pool

In [20]:
# ============================================================
# STEP 1 - CELL 8
# AMAZON PRODUCT CANDIDATE POOL
# ============================================================

candidate_mask = (
    amazon_products["parent_asin"].notna()
    &
    amazon_products["title"].notna()
    &
    amazon_products["title"].ne("")
    &
    amazon_products["price"].notna()
    &
    (amazon_products["price"] > 0)
    &
    amazon_products["average_rating"].notna()
    &
    (amazon_products["average_rating"] > 0)
    &
    (amazon_products["rating_number"] >= 10)
)

amazon_candidates = (
    amazon_products[
        candidate_mask
    ]
    .copy()
    .reset_index(drop=True)
)


print("=" * 70)
print("AMAZON CANDIDATE PRODUCTS")
print("=" * 70)

print(
    "Products:",
    f"{len(amazon_candidates):,}"
)

print(
    "Unique ASIN:",
    f"{amazon_candidates['parent_asin'].nunique():,}"
)

print("\nPrice:")
print(
    amazon_candidates["price"]
    .describe()
)

print("\nRating number:")
print(
    amazon_candidates["rating_number"]
    .describe()
)

AMAZON CANDIDATE PRODUCTS
Products: 335,560
Unique ASIN: 335,560

Price:
count    335560.000000
mean         84.988035
std         244.649695
min           0.010000
25%          11.990000
50%          21.990000
75%          59.980000
max       24996.990000
Name: price, dtype: float64

Rating number:
count    335560.000000
mean        506.894201
std        3954.987598
min          10.000000
25%          23.000000
50%          59.000000
75%         208.000000
max      585624.000000
Name: rating_number, dtype: float64


# Kiểm tra Review ↔ Metadata

In [21]:
# ============================================================
# STEP 1 - CELL 9
# AMAZON REVIEW ↔ PRODUCT MATCH
# ============================================================

print("=" * 70)
print("AMAZON REVIEW ↔ PRODUCT MATCH")
print("=" * 70)


candidate_ids = set(
    amazon_candidates["parent_asin"]
    .dropna()
)


amazon_reviews["candidate_match"] = (
    amazon_reviews["parent_asin"]
    .isin(candidate_ids)
)


# ------------------------------------------------------------
# Review-level match
# ------------------------------------------------------------

matched_reviews = int(
    amazon_reviews["candidate_match"].sum()
)

review_match_rate = (
    matched_reviews /
    len(amazon_reviews) *
    100
)


print("\n[1] REVIEW MATCH")
print("-" * 70)

print(
    "Total reviews      :",
    f"{len(amazon_reviews):,}"
)

print(
    "Matched reviews    :",
    f"{matched_reviews:,}"
)

print(
    "Review match rate  :",
    f"{review_match_rate:.2f}%"
)


# ------------------------------------------------------------
# Product-level match
# ------------------------------------------------------------

review_product_ids = set(
    amazon_reviews["parent_asin"]
    .dropna()
)

matched_product_ids = (
    review_product_ids
    & candidate_ids
)


print("\n[2] PRODUCT MATCH")
print("-" * 70)

print(
    "Products in review shard :",
    f"{len(review_product_ids):,}"
)

print(
    "Candidate products       :",
    f"{len(candidate_ids):,}"
)

print(
    "Matched products         :",
    f"{len(matched_product_ids):,}"
)


# ------------------------------------------------------------
# Keep candidate reviews
# ------------------------------------------------------------

amazon_matched_reviews = (
    amazon_reviews[
        amazon_reviews["candidate_match"]
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Interaction statistics
# ------------------------------------------------------------

print("\n[3] INTERACTION STATISTICS")
print("-" * 70)

print(
    "Interactions:",
    f"{len(amazon_matched_reviews):,}"
)

print(
    "Users:",
    f"{amazon_matched_reviews['user_id'].nunique():,}"
)

print(
    "Products:",
    f"{amazon_matched_reviews['parent_asin'].nunique():,}"
)


# ------------------------------------------------------------
# User interaction counts
# ------------------------------------------------------------

user_counts = (
    amazon_matched_reviews
    .groupby("user_id")
    .size()
)


print("\n[4] USER ACTIVITY")
print("-" * 70)

for threshold in [
    1,
    2,
    3,
    5,
    10,
    20
]:

    count = (
        user_counts >= threshold
    ).sum()

    print(
        f"Users with >= {threshold:>2} interactions: "
        f"{count:>10,}"
    )


# ------------------------------------------------------------
# Product interaction counts
# ------------------------------------------------------------

product_counts = (
    amazon_matched_reviews
    .groupby("parent_asin")
    .size()
)


print("\n[5] PRODUCT ACTIVITY")
print("-" * 70)

for threshold in [
    1,
    2,
    5,
    10,
    20,
    50,
    100
]:

    count = (
        product_counts >= threshold
    ).sum()

    print(
        f"Products with >= {threshold:>3} interactions: "
        f"{count:>10,}"
    )


# ------------------------------------------------------------
# Sparsity
# ------------------------------------------------------------

n_users = (
    amazon_matched_reviews["user_id"]
    .nunique()
)

n_products = (
    amazon_matched_reviews["parent_asin"]
    .nunique()
)

n_interactions = len(
    amazon_matched_reviews
)

if n_users > 0 and n_products > 0:

    density = (
        n_interactions /
        (n_users * n_products)
    )

    sparsity = (
        1 - density
    ) * 100

else:

    density = 0
    sparsity = 0


print("\n[6] MATRIX SPARSITY")
print("-" * 70)

print(
    "Users       :",
    f"{n_users:,}"
)

print(
    "Products    :",
    f"{n_products:,}"
)

print(
    "Interactions:",
    f"{n_interactions:,}"
)

print(
    "Density     :",
    f"{density:.8f}"
)

print(
    "Sparsity    :",
    f"{sparsity:.6f}%"
)


print("\n" + "=" * 70)
print("✅ AMAZON LINKAGE AUDIT COMPLETED")
print("=" * 70)

AMAZON REVIEW ↔ PRODUCT MATCH

[1] REVIEW MATCH
----------------------------------------------------------------------
Total reviews      : 1,290,793
Matched reviews    : 736,093
Review match rate  : 57.03%

[2] PRODUCT MATCH
----------------------------------------------------------------------
Products in review shard : 315,544
Candidate products       : 335,560
Matched products         : 121,917

[3] INTERACTION STATISTICS
----------------------------------------------------------------------
Interactions: 736,093
Users: 198,427
Products: 121,917

[4] USER ACTIVITY
----------------------------------------------------------------------
Users with >=  1 interactions:    198,427
Users with >=  2 interactions:    117,089
Users with >=  3 interactions:     77,578
Users with >=  5 interactions:     41,313
Users with >= 10 interactions:     14,577
Users with >= 20 interactions:      4,067

[5] PRODUCT ACTIVITY
----------------------------------------------------------------------
Products 

# Tạo Amazon dataset chuẩn

In [22]:
# ============================================================
# STEP 1 - CELL 10
# BUILD CLEAN AMAZON DATA
# ============================================================

print("=" * 70)
print("BUILD CLEAN AMAZON DATA")
print("=" * 70)

# ------------------------------------------------------------
# 1. Keep matched reviews
# ------------------------------------------------------------

amazon_reviews_clean = (
    amazon_matched_reviews
    .copy()
)

# Remove helper column
amazon_reviews_clean.drop(
    columns=["candidate_match"],
    errors="ignore",
    inplace=True
)

# ------------------------------------------------------------
# 2. Products actually appearing in reviews
# ------------------------------------------------------------

active_amazon_ids = set(
    amazon_reviews_clean["parent_asin"]
    .dropna()
    .unique()
)

amazon_products_clean = (
    amazon_candidates[
        amazon_candidates["parent_asin"]
        .isin(active_amazon_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Add source information
# ------------------------------------------------------------

amazon_products_clean["source"] = "amazon"
amazon_products_clean["language"] = "en"
amazon_products_clean["currency"] = "USD"

amazon_reviews_clean["source"] = "amazon"
amazon_reviews_clean["language"] = "en"

# ------------------------------------------------------------
# 4. Create globally unique IDs
# ------------------------------------------------------------

amazon_products_clean["product_id"] = (
    "amazon_" +
    amazon_products_clean["parent_asin"].astype(str)
)

amazon_reviews_clean["product_id"] = (
    "amazon_" +
    amazon_reviews_clean["parent_asin"].astype(str)
)

amazon_reviews_clean["user_id"] = (
    "amazon_" +
    amazon_reviews_clean["user_id"].astype(str)
)

# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("\nAmazon products :", f"{len(amazon_products_clean):,}")
print("Amazon reviews  :", f"{len(amazon_reviews_clean):,}")
print(
    "Amazon users    :",
    f"{amazon_reviews_clean['user_id'].nunique():,}"
)

print(
    "Product IDs     :",
    f"{amazon_products_clean['product_id'].nunique():,}"
)

print("\nProduct sample:")
display(
    amazon_products_clean[
        [
            "product_id",
            "title",
            "main_category",
            "price",
            "currency",
            "average_rating",
            "rating_number"
        ]
    ].head()
)

print("\nReview sample:")
display(
    amazon_reviews_clean[
        [
            "user_id",
            "product_id",
            "rating",
            "text",
            "timestamp"
        ]
    ].head()
)

BUILD CLEAN AMAZON DATA

Amazon products : 121,917
Amazon reviews  : 736,093
Amazon users    : 198,427
Product IDs     : 121,917

Product sample:


,product_id,title,main_category,price,currency,average_rating,rating_number
0,amazon_B07PXGQC1Q,Apple AirPods (2nd Generation) Wireless Earbuds with Lightning Charging Case Included. Over 24 H...,Apple Products,99.00,USD,4.7,585624
1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",Amazon Devices,24.99,USD,4.7,542825
2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",Home Audio & Theater,8.55,USD,4.7,507202
3,amazon_B08CLNX58K,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT,Computers,13.40,USD,4.7,402800
4,amazon_B08WJSHSLC,Fire TV Stick (3rd Gen) with Alexa Voice Remote (includes TV controls) + Star Wars The Mandalori...,Amazon Devices,44.98,USD,4.7,379565



Review sample:


,user_id,product_id,rating,text,timestamp
0,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B07N69T6TM,1.0,"These didn’t work. Idk if they were damaged in shipping or what, but the lenses were loose or so...",1592678549731
1,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B01G8JO5F2,5.0,I love these. They even come with a carry case and several sizes of ear bud inserts. Thank heave...,1523093017534
2,amazon_AG2L7H23R5LLKDKLBEF2Q3L2MVDA,amazon_B07CJYMRWM,5.0,I've bought these headphones three times because I love them so much and overuse them and find w...,1676601581238
3,amazon_AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,amazon_B07CML419K,5.0,"Light weight, quiet and totally awesome!!! It doesn’t come with a charging base, but the cord en...",1637522881041
4,amazon_AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,amazon_B09S6Y5BRG,5.0,These are fantastic headphones and I love that i can have my cell phone plugged in at a central ...,1541356831659


# LOAD & STANDARDIZE TIKI PRODUCTS


In [23]:
# ============================================================
# STEP 1 - CELL 11
# LOAD & STANDARDIZE TIKI PRODUCTS
# ============================================================

print("=" * 70)
print("LOAD TIKI PRODUCT DATA")
print("=" * 70)

tiki_parts = []

for path in tiki_files:

    temp = pd.read_csv(
        path,
        low_memory=False
    )

    temp["source_file"] = path.name

    tiki_parts.append(temp)

    print(
        f"{path.name:<60}"
        f"{len(temp):>8,}"
    )

tiki_raw = pd.concat(
    tiki_parts,
    ignore_index=True
)

del tiki_parts
gc.collect()

print("\nRaw Tiki rows:", f"{len(tiki_raw):,}")
print("Columns:", tiki_raw.columns.tolist())

LOAD TIKI PRODUCT DATA
vietnamese_tiki_products_backpacks_suitcases.csv               5,361
vietnamese_tiki_products_fashion_accessories.csv              16,019
vietnamese_tiki_products_men_bags.csv                          4,234
vietnamese_tiki_products_men_shoes.csv                         5,745
vietnamese_tiki_products_women_bags.csv                        4,325
vietnamese_tiki_products_women_shoes.csv                       5,919

Raw Tiki rows: 41,603
Columns: ['Unnamed: 0', 'id', 'name', 'description', 'original_price', 'price', 'fulfillment_type', 'brand', 'review_count', 'rating_average', 'favourite_count', 'pay_later', 'current_seller', 'date_created', 'number_of_images', 'vnd_cashback', 'has_video', 'category', 'quantity_sold', 'source_file']


# CLEAN TIKI PRODUCTS


In [24]:
# ============================================================
# STEP 1 - CELL 12
# CLEAN TIKI PRODUCTS
# ============================================================

print("=" * 70)
print("CLEAN TIKI PRODUCTS")
print("=" * 70)

tiki_products_clean = tiki_raw.copy()

# ------------------------------------------------------------
# ID
# ------------------------------------------------------------

tiki_products_clean["id"] = (
    tiki_products_clean["id"]
    .astype("string")
    .str.strip()
)

# ------------------------------------------------------------
# Text
# ------------------------------------------------------------

for col in [
    "name",
    "description",
    "brand",
    "category"
]:
    if col in tiki_products_clean.columns:

        tiki_products_clean[col] = (
            tiki_products_clean[col]
            .astype("string")
            .str.strip()
        )

# ------------------------------------------------------------
# Numeric
# ------------------------------------------------------------

for col in [
    "price",
    "review_count",
    "rating_average",
    "quantity_sold"
]:
    if col in tiki_products_clean.columns:

        tiki_products_clean[col] = pd.to_numeric(
            tiki_products_clean[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# Basic validity
# ------------------------------------------------------------

valid_tiki = (
    tiki_products_clean["id"].notna()
    &
    tiki_products_clean["name"].notna()
    &
    tiki_products_clean["name"].ne("")
    &
    tiki_products_clean["price"].notna()
    &
    (tiki_products_clean["price"] > 0)
)

tiki_products_clean = (
    tiki_products_clean[
        valid_tiki
    ]
    .copy()
)

# ------------------------------------------------------------
# Duplicate IDs
# Keep row with more review information
# ------------------------------------------------------------

tiki_products_clean = (
    tiki_products_clean
    .sort_values(
        [
            "review_count",
            "quantity_sold"
        ],
        ascending=False
    )
    .drop_duplicates(
        subset=["id"],
        keep="first"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Global IDs
# ------------------------------------------------------------

tiki_products_clean["product_id"] = (
    "tiki_" +
    tiki_products_clean["id"].astype(str)
)

tiki_products_clean["source"] = "tiki"
tiki_products_clean["language"] = "vi"
tiki_products_clean["currency"] = "VND"

print(
    "\nClean Tiki products:",
    f"{len(tiki_products_clean):,}"
)

print(
    "Unique IDs:",
    f"{tiki_products_clean['product_id'].nunique():,}"
)

print("\nSample:")

display(
    tiki_products_clean[
        [
            "product_id",
            "name",
            "category",
            "brand",
            "price",
            "currency",
            "rating_average",
            "review_count",
            "quantity_sold"
        ]
    ].head()
)

CLEAN TIKI PRODUCTS

Clean Tiki products: 41,575
Unique IDs: 41,575

Sample:


,product_id,name,category,brand,price,currency,rating_average,review_count,quantity_sold
0,tiki_72570436,Dép Nam Quai Ngang Kẻ SUPERSTAR DP65,Giày - Dép nam,Laceva,129000,VND,4.6,969,2894
1,tiki_52758433,"Combo 5 Đôi Tất Nam Vớ Nam Cổ Dài Sợi Cotton Cao Cấp Chống Hôi Chân, Không Xù, Không Bai Dão (Mi...","Tất, vớ nam",JM Jamano,95000,VND,4.7,966,3473
2,tiki_257326416,Keo Dán Giày Nhiệt XIMO Trong Suốt Siêu Dính Dùng Không Tổn Thương Da XKDG06,Giày - Dép nam,Ximo,35000,VND,4.6,926,5429
3,tiki_52758100,"Combo 5 Đôi Tất Nam Vớ Nam Cổ Dài Sợi Cotton Cao Cấp Chống Hôi Chân, Không Xù, Không Bai Dão (Cù...","Tất, vớ nam",JM Jamano,95000,VND,4.8,799,3633
4,tiki_28590683,Miếng lót giảm size cho giày bị rộng Cao Cấp - buybox - BBPK11,Thời Trang,buybox,70000,VND,4.5,582,9245


# LOAD VIETNAMESE / SHOPEE REVIEWS


In [25]:
# ============================================================
# STEP 1 - CELL 13
# LOAD OLD VIETNAMESE / SHOPEE REVIEWS
# ============================================================

print("=" * 70)
print("LOAD VIETNAMESE REVIEW DATA")
print("=" * 70)

if len(old_vietnamese_files) != 1:
    raise ValueError(
        "Expected exactly one vietnamese_reviews.csv"
    )

vi_path = old_vietnamese_files[0]

shopee_reviews = pd.read_csv(
    vi_path,
    low_memory=False
)

print("File:", vi_path)

print(
    "Rows:",
    f"{len(shopee_reviews):,}"
)

print(
    "Columns:",
    shopee_reviews.columns.tolist()
)

display(
    shopee_reviews.head()
)

LOAD VIETNAMESE REVIEW DATA
File: /kaggle/input/datasets/trnquangtriu/1bdataset/vietnamese_reviews.csv
Rows: 31,086
Columns: ['user_id', 'product_id', 'rating', 'raw_review', 'review_text', 'timestamp', 'product_name', 'product_image', 'shop_id', 'category', 'subcategory', 'language', 'source']


,user_id,product_id,rating,raw_review,review_text,timestamp,product_name,product_image,shop_id,category,subcategory,language,source
0,vi_73180065,vi_2109743537,5,Hàng rẻ mà chất lượng đấy chứ. Cho shop 5 sao. Trước mua điều khiển điều hoà shop này dùng vẫn t...,Hàng rẻ mà chất lượng đấy chứ. Cho shop 5 sao. Trước mua điều khiển điều hoà shop này dùng vẫn t...,1566628842000000000,Remote Điều Khiển TV SMART SAMSUNG L1088+ [LOẠI ĐẸP - Giá Sỉ] Đa Năng Sử Dụng được tất cả TV SAM...,https://down-bs-us.img.susercontent.com/vn-11134207-7qukw-lkewpwahrmi0fc.webp,64962549,Electronic,ThietBiDienTu,vi,shopee_vi
1,vi_12965235,vi_2712611171,5,Shop giao hàng nhanh đóng gói rất đẹp. Máy chơi game hịn hò quay lại tuổi thơ. Mức giá cả quá ư ...,Shop giao hàng nhanh đóng gói rất đẹp. Máy chơi game hịn hò quay lại tuổi thơ. Mức giá cả quá ư ...,1567567549000000000,"Máy chơi game cầm tay SUP 400 in 1, máy điện tử 4 nút cổ điển",https://down-bs-us.img.susercontent.com/88e244f6e72bae4b4ec817bee0b57d41.webp,16345337,Electronic,ThietBiDienTu,vi,shopee_vi
2,vi_144810899,vi_2712611171,5,Vote 5⭐⭐⭐⭐⭐ cho shop luôn. Giao hàng max nhanh. Máy chơi game cầm vừa tay. Giá lại rẻ. Đóng hàng...,Vote 5⭐⭐⭐⭐⭐ cho shop luôn. Giao hàng max nhanh. Máy chơi game cầm vừa tay. Giá lại rẻ. Đóng hàng...,1567569063000000000,"Máy chơi game cầm tay SUP 400 in 1, máy điện tử 4 nút cổ điển",https://down-bs-us.img.susercontent.com/88e244f6e72bae4b4ec817bee0b57d41.webp,16345337,Electronic,ThietBiDienTu,vi,shopee_vi
3,vi_29682369,vi_2712611171,5,"Máy màu vàng siêu đáng iêu, đóng gói hàng cẩn cận.Thời gian giao hàng rất nhanh. Tặng shop ngàn ...","Máy màu vàng siêu đáng iêu, đóng gói hàng cẩn cận.Thời gian giao hàng rất nhanh. Tặng shop ngàn ...",1568012318000000000,"Máy chơi game cầm tay SUP 400 in 1, máy điện tử 4 nút cổ điển",https://down-bs-us.img.susercontent.com/88e244f6e72bae4b4ec817bee0b57d41.webp,16345337,Electronic,ThietBiDienTu,vi,shopee_vi
4,vi_43083074,vi_1472134609,5,"Sp chắc chắn và không bị sắc cạnh, sử dụng rất thích. Rất tiện lợi cho những người đang bị bó bộ...","Sp chắc chắn và không bị sắc cạnh, sử dụng rất thích. Rất tiện lợi cho những người đang bị bó bộ...",1569304270000000000,Cây Gãi Lưng Inox Kéo Dài Rút Gọn 50Cm - Tay Dụng Cụ Đồ Gãi Ngứa Siêu Đỉnh,https://down-bs-us.img.susercontent.com/vn-11134201-7r98o-lynjextmmij143.webp,13350150,HealthBeauty,DungCuMassageTriLieu,vi,shopee_vi


# CLEAN SHOPEE REVIEW DATA


In [26]:
# ============================================================
# STEP 1 - CELL 14
# CLEAN SHOPEE REVIEW DATA
# ============================================================

print("=" * 70)
print("CLEAN SHOPEE REVIEWS")
print("=" * 70)

shopee_reviews_clean = shopee_reviews.copy()

# ------------------------------------------------------------
# Basic IDs
# ------------------------------------------------------------

shopee_reviews_clean["user_id"] = (
    shopee_reviews_clean["user_id"]
    .astype("string")
    .str.strip()
)

shopee_reviews_clean["product_id"] = (
    shopee_reviews_clean["product_id"]
    .astype("string")
    .str.strip()
)

# Old pipeline used vi_ prefix.
# Remove it before creating new explicit source prefix.
shopee_reviews_clean["product_id"] = (
    shopee_reviews_clean["product_id"]
    .str.replace(
        r"^vi_",
        "",
        regex=True
    )
)

shopee_reviews_clean["user_id"] = (
    shopee_reviews_clean["user_id"]
    .str.replace(
        r"^vi_",
        "",
        regex=True
    )
)

# ------------------------------------------------------------
# Rating
# ------------------------------------------------------------

shopee_reviews_clean["rating"] = pd.to_numeric(
    shopee_reviews_clean["rating"],
    errors="coerce"
)

# ------------------------------------------------------------
# Review text
# ------------------------------------------------------------

if "review_text" in shopee_reviews_clean.columns:

    shopee_reviews_clean["review_text"] = (
        shopee_reviews_clean["review_text"]
        .astype("string")
        .str.strip()
    )

# ------------------------------------------------------------
# Valid interaction
# ------------------------------------------------------------

valid_shopee = (
    shopee_reviews_clean["user_id"].notna()
    &
    shopee_reviews_clean["product_id"].notna()
    &
    shopee_reviews_clean["rating"].between(1, 5)
)

shopee_reviews_clean = (
    shopee_reviews_clean[
        valid_shopee
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# New global IDs
# ------------------------------------------------------------

shopee_reviews_clean["user_id"] = (
    "shopee_" +
    shopee_reviews_clean["user_id"].astype(str)
)

shopee_reviews_clean["product_id"] = (
    "shopee_" +
    shopee_reviews_clean["product_id"].astype(str)
)

shopee_reviews_clean["source"] = "shopee"
shopee_reviews_clean["language"] = "vi"

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(
    "Reviews :",
    f"{len(shopee_reviews_clean):,}"
)

print(
    "Users   :",
    f"{shopee_reviews_clean['user_id'].nunique():,}"
)

print(
    "Products:",
    f"{shopee_reviews_clean['product_id'].nunique():,}"
)

display(
    shopee_reviews_clean.head()
)

CLEAN SHOPEE REVIEWS
Reviews : 31,086
Users   : 29,888
Products: 1,434


,user_id,product_id,rating,raw_review,review_text,timestamp,product_name,product_image,shop_id,category,subcategory,language,source
0,shopee_73180065,shopee_2109743537,5,Hàng rẻ mà chất lượng đấy chứ. Cho shop 5 sao. Trước mua điều khiển điều hoà shop này dùng vẫn t...,Hàng rẻ mà chất lượng đấy chứ. Cho shop 5 sao. Trước mua điều khiển điều hoà shop này dùng vẫn t...,1566628842000000000,Remote Điều Khiển TV SMART SAMSUNG L1088+ [LOẠI ĐẸP - Giá Sỉ] Đa Năng Sử Dụng được tất cả TV SAM...,https://down-bs-us.img.susercontent.com/vn-11134207-7qukw-lkewpwahrmi0fc.webp,64962549,Electronic,ThietBiDienTu,vi,shopee
1,shopee_12965235,shopee_2712611171,5,Shop giao hàng nhanh đóng gói rất đẹp. Máy chơi game hịn hò quay lại tuổi thơ. Mức giá cả quá ư ...,Shop giao hàng nhanh đóng gói rất đẹp. Máy chơi game hịn hò quay lại tuổi thơ. Mức giá cả quá ư ...,1567567549000000000,"Máy chơi game cầm tay SUP 400 in 1, máy điện tử 4 nút cổ điển",https://down-bs-us.img.susercontent.com/88e244f6e72bae4b4ec817bee0b57d41.webp,16345337,Electronic,ThietBiDienTu,vi,shopee
2,shopee_144810899,shopee_2712611171,5,Vote 5⭐⭐⭐⭐⭐ cho shop luôn. Giao hàng max nhanh. Máy chơi game cầm vừa tay. Giá lại rẻ. Đóng hàng...,Vote 5⭐⭐⭐⭐⭐ cho shop luôn. Giao hàng max nhanh. Máy chơi game cầm vừa tay. Giá lại rẻ. Đóng hàng...,1567569063000000000,"Máy chơi game cầm tay SUP 400 in 1, máy điện tử 4 nút cổ điển",https://down-bs-us.img.susercontent.com/88e244f6e72bae4b4ec817bee0b57d41.webp,16345337,Electronic,ThietBiDienTu,vi,shopee
3,shopee_29682369,shopee_2712611171,5,"Máy màu vàng siêu đáng iêu, đóng gói hàng cẩn cận.Thời gian giao hàng rất nhanh. Tặng shop ngàn ...","Máy màu vàng siêu đáng iêu, đóng gói hàng cẩn cận.Thời gian giao hàng rất nhanh. Tặng shop ngàn ...",1568012318000000000,"Máy chơi game cầm tay SUP 400 in 1, máy điện tử 4 nút cổ điển",https://down-bs-us.img.susercontent.com/88e244f6e72bae4b4ec817bee0b57d41.webp,16345337,Electronic,ThietBiDienTu,vi,shopee
4,shopee_43083074,shopee_1472134609,5,"Sp chắc chắn và không bị sắc cạnh, sử dụng rất thích. Rất tiện lợi cho những người đang bị bó bộ...","Sp chắc chắn và không bị sắc cạnh, sử dụng rất thích. Rất tiện lợi cho những người đang bị bó bộ...",1569304270000000000,Cây Gãi Lưng Inox Kéo Dài Rút Gọn 50Cm - Tay Dụng Cụ Đồ Gãi Ngứa Siêu Đỉnh,https://down-bs-us.img.susercontent.com/vn-11134201-7r98o-lynjextmmij143.webp,13350150,HealthBeauty,DungCuMassageTriLieu,vi,shopee


# BUILD SHOPEE PRODUCT CATALOG


In [27]:
# ============================================================
# STEP 1 - CELL 15
# BUILD SHOPEE PRODUCT CATALOG
# ============================================================

print("=" * 70)
print("BUILD SHOPEE PRODUCT CATALOG")
print("=" * 70)

product_cols = [
    "product_id",
    "product_name",
    "product_image",
    "shop_id",
    "category",
    "subcategory"
]

available_product_cols = [
    c for c in product_cols
    if c in shopee_reviews_clean.columns
]

shopee_products_clean = (
    shopee_reviews_clean[
        available_product_cols
    ]
    .drop_duplicates(
        subset=["product_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

# Product-level rating statistics
shopee_stats = (
    shopee_reviews_clean
    .groupby("product_id")
    .agg(
        average_rating=("rating", "mean"),
        rating_number=("rating", "size")
    )
    .reset_index()
)

shopee_products_clean = (
    shopee_products_clean
    .merge(
        shopee_stats,
        on="product_id",
        how="left"
    )
)

shopee_products_clean["source"] = "shopee"
shopee_products_clean["language"] = "vi"
shopee_products_clean["currency"] = "VND"

# Price unavailable in this source
shopee_products_clean["price"] = np.nan

print(
    "Shopee products:",
    f"{len(shopee_products_clean):,}"
)

display(
    shopee_products_clean.head()
)

BUILD SHOPEE PRODUCT CATALOG
Shopee products: 1,434


,product_id,product_name,product_image,shop_id,category,subcategory,average_rating,rating_number,source,language,currency,price
0,shopee_2109743537,Remote Điều Khiển TV SMART SAMSUNG L1088+ [LOẠI ĐẸP - Giá Sỉ] Đa Năng Sử Dụng được tất cả TV SAM...,https://down-bs-us.img.susercontent.com/vn-11134207-7qukw-lkewpwahrmi0fc.webp,64962549,Electronic,ThietBiDienTu,4.818182,22,shopee,vi,VND,NaN
1,shopee_2712611171,"Máy chơi game cầm tay SUP 400 in 1, máy điện tử 4 nút cổ điển",https://down-bs-us.img.susercontent.com/88e244f6e72bae4b4ec817bee0b57d41.webp,16345337,Electronic,ThietBiDienTu,5.000000,24,shopee,vi,VND,NaN
2,shopee_1472134609,Cây Gãi Lưng Inox Kéo Dài Rút Gọn 50Cm - Tay Dụng Cụ Đồ Gãi Ngứa Siêu Đỉnh,https://down-bs-us.img.susercontent.com/vn-11134201-7r98o-lynjextmmij143.webp,13350150,HealthBeauty,DungCuMassageTriLieu,4.954545,22,shopee,vi,VND,NaN
3,shopee_1050114357,Dép xỏ ngón nam chất liệu xốp cao cấp Latoma TA1091 (Nhiều màu),https://down-bs-us.img.susercontent.com/vn-11134207-7r98o-llbktw0hpdx259.webp,48928826,Fashion,GiayDepNam,5.000000,23,shopee,vi,VND,NaN
4,shopee_2266180117,Bình lọc nước gia đình,https://down-bs-us.img.susercontent.com/vn-11134207-7qukw-lijlm4jkkdzm2e.webp,20242681,HomeLifestyle,DoGiaDung,4.909091,22,shopee,vi,VND,NaN


# Chuẩn hóa products.csv

In [28]:
# ============================================================
# STEP 1 - CELL 16
# BUILD UNIFIED PRODUCT TABLE
# ============================================================

print("=" * 70)
print("BUILD UNIFIED PRODUCT TABLE")
print("=" * 70)


# ============================================================
# AMAZON
# ============================================================

amazon_final_products = pd.DataFrame({

    "product_id":
        amazon_products_clean["product_id"],

    "product_name":
        amazon_products_clean["title"],

    "category":
        amazon_products_clean["main_category"],

    "subcategory":
        pd.NA,

    "brand":
        amazon_products_clean["store"],

    "price":
        amazon_products_clean["price"],

    "currency":
        "USD",

    "average_rating":
        amazon_products_clean["average_rating"],

    "review_count":
        amazon_products_clean["rating_number"],

    "sold_count":
        np.nan,

    "description":
        pd.NA,

    "image_url":
        pd.NA,

    "product_url":
        pd.NA,

    "source":
        "amazon",

    "language":
        "en"
})


# ============================================================
# TIKI
# ============================================================

tiki_final_products = pd.DataFrame({

    "product_id":
        tiki_products_clean["product_id"],

    "product_name":
        tiki_products_clean["name"],

    "category":
        tiki_products_clean["category"],

    "subcategory":
        pd.NA,

    "brand":
        tiki_products_clean["brand"],

    "price":
        tiki_products_clean["price"],

    "currency":
        "VND",

    "average_rating":
        tiki_products_clean["rating_average"],

    "review_count":
        tiki_products_clean["review_count"],

    "sold_count":
        tiki_products_clean["quantity_sold"],

    "description":
        tiki_products_clean["description"],

    "image_url":
        pd.NA,

    "product_url":
        pd.NA,

    "source":
        "tiki",

    "language":
        "vi"
})


# ============================================================
# SHOPEE
# ============================================================

shopee_final_products = pd.DataFrame({

    "product_id":
        shopee_products_clean["product_id"],

    "product_name":
        shopee_products_clean.get(
            "product_name",
            pd.Series(pd.NA, index=shopee_products_clean.index)
        ),

    "category":
        shopee_products_clean.get(
            "category",
            pd.Series(pd.NA, index=shopee_products_clean.index)
        ),

    "subcategory":
        shopee_products_clean.get(
            "subcategory",
            pd.Series(pd.NA, index=shopee_products_clean.index)
        ),

    "brand":
        pd.NA,

    "price":
        np.nan,

    "currency":
        "VND",

    "average_rating":
        shopee_products_clean["average_rating"],

    "review_count":
        shopee_products_clean["rating_number"],

    "sold_count":
        np.nan,

    "description":
        pd.NA,

    "image_url":
        shopee_products_clean.get(
            "product_image",
            pd.Series(pd.NA, index=shopee_products_clean.index)
        ),

    "product_url":
        pd.NA,

    "source":
        "shopee",

    "language":
        "vi"
})


# ============================================================
# CONCAT
# ============================================================

products = pd.concat(
    [
        amazon_final_products,
        tiki_final_products,
        shopee_final_products
    ],
    ignore_index=True
)

print(
    "\nTotal products:",
    f"{len(products):,}"
)

print("\nProducts by source:")

print(
    products["source"]
    .value_counts()
)

print(
    "\nDuplicate product_id:",
    products["product_id"]
    .duplicated()
    .sum()
)

display(
    products.sample(
        min(10, len(products)),
        random_state=42
    )
)

BUILD UNIFIED PRODUCT TABLE

Total products: 164,926

Products by source:
source
amazon    121917
tiki       41575
shopee      1434
Name: count, dtype: int64

Duplicate product_id: 0


,product_id,product_name,category,subcategory,brand,price,currency,average_rating,review_count,sold_count,description,image_url,product_url,source,language
47747,amazon_B072MLBMBF,"mCover iPearl Hard Shell Case for 2017 11.6"" Dell Chromebook 11 3189 Series 2-in-1 Laptop (NOT C...",Computers,NaN,mCover,19.99,USD,4.3,354,NaN,<NA>,<NA>,<NA>,amazon,en
124900,tiki_124778123,Dép tổ ong Vàng thái loại xịn cho nam,Dép nam quai ngang,NaN,OEM,34999.00,VND,4.8,8,40.0,Dép tổ ong là loại dép ai cũng biết rồi đúng không cả nhà . Đi mấy năm chẳng hỏng một đôi ấy mà ...,<NA>,<NA>,tiki,vi
126307,tiki_44324447,Túi xách da bò công sở cao cấp phong cách châu âu TX DA BÒ 7612,Thời Trang,NaN,ANANSHOP688,801000.00,VND,4.8,5,15.0,"Kích thước : rộng 38cm ,cao 28cm,dày 10cmPhong cách : châu âuMàu sắc : như hìnhPhương pháp sản x...",<NA>,<NA>,tiki,vi
95258,amazon_B07C2C7ZSB,Mission Darkness NeoLok Non-Window Faraday Bag for Tablets (+ Secure Magnetic Closure) // Device...,Computers,NaN,Mission Darkness,65.00,USD,4.5,59,NaN,<NA>,<NA>,<NA>,amazon,en
14211,amazon_B08HCLB1JZ,"LP-E10 Battery Charger Pack, LP 2-Pack Battery & Charger, Compatible with Canon EOS Rebel T7, T6...",Camera & Photo,NaN,LP,26.98,USD,4.6,1792,NaN,<NA>,<NA>,<NA>,amazon,en
10003,amazon_B07VBKH9NH,"Messenger Bag for Men,VASCHY Vintage Water Resistant Canvas Satchel 14 15.6 17inch Laptop Briefcase",AMAZON FASHION,NaN,VASCHY,46.99,USD,4.7,2597,NaN,<NA>,<NA>,<NA>,amazon,en
6059,amazon_B07RT577T7,9inch Truck GPS Big Touchscreen Trucking GPS XGODY GPS Navigation for car Navigation 8GB ROM SAT...,All Electronics,NaN,Xgody,92.99,USD,3.6,4226,NaN,<NA>,<NA>,<NA>,amazon,en
127853,tiki_26210394,Hình Xăm Dán Đầu Lâu QSA025,Thời Trang,NaN,OEM,50000.00,VND,5.0,3,11.0,Thông tin sản phẩm:Hình xăm dán là sản phẩm giúp nổi bật cá tính riêng của mỗi ngườiCó rất nhiều...,<NA>,<NA>,tiki,vi
135264,tiki_36756069,TÚI TOTE ĐEO CHÉO CHÚ VỊT OM0059,Root,NaN,OEM,107000.00,VND,0.0,0,13.0,Túi tote là một loại túi lớn có dây xách song song nhau nối từ hai bên thành túi.Tote nguyên mẫu...,<NA>,<NA>,tiki,vi
111570,amazon_B00563LXDY,"Intel Socket 775 Copper Core/Aluminum Heat Sink & 3.5"" Ball Bearing Fan w/4-Pin Connector up to ...",Computers,NaN,Intel,29.97,USD,4.4,25,NaN,<NA>,<NA>,<NA>,amazon,en


# Chuẩn hóa reviews.csv

In [29]:
# ============================================================
# STEP 1 - CELL 17
# BUILD UNIFIED REVIEW TABLE
# ============================================================

print("=" * 70)
print("BUILD UNIFIED REVIEW TABLE")
print("=" * 70)


# ------------------------------------------------------------
# Amazon
# ------------------------------------------------------------

amazon_final_reviews = pd.DataFrame({

    "user_id":
        amazon_reviews_clean["user_id"],

    "product_id":
        amazon_reviews_clean["product_id"],

    "rating":
        amazon_reviews_clean["rating"],

    "review_title":
        amazon_reviews_clean.get(
            "title",
            pd.Series(pd.NA, index=amazon_reviews_clean.index)
        ),

    "review_text":
        amazon_reviews_clean["text"],

    "timestamp":
        amazon_reviews_clean["timestamp"],

    "verified_purchase":
        amazon_reviews_clean.get(
            "verified_purchase",
            pd.Series(pd.NA, index=amazon_reviews_clean.index)
        ),

    "helpful_vote":
        amazon_reviews_clean.get(
            "helpful_vote",
            pd.Series(pd.NA, index=amazon_reviews_clean.index)
        ),

    "language":
        "en",

    "source":
        "amazon"
})


# ------------------------------------------------------------
# Shopee
# ------------------------------------------------------------

shopee_final_reviews = pd.DataFrame({

    "user_id":
        shopee_reviews_clean["user_id"],

    "product_id":
        shopee_reviews_clean["product_id"],

    "rating":
        shopee_reviews_clean["rating"],

    "review_title":
        pd.NA,

    "review_text":
        shopee_reviews_clean["review_text"],

    "timestamp":
        shopee_reviews_clean["timestamp"],

    "verified_purchase":
        pd.NA,

    "helpful_vote":
        np.nan,

    "language":
        "vi",

    "source":
        "shopee"
})


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

reviews = pd.concat(
    [
        amazon_final_reviews,
        shopee_final_reviews
    ],
    ignore_index=True
)

reviews.insert(
    0,
    "review_id",
    [
        f"review_{i:07d}"
        for i in range(len(reviews))
    ]
)


print(
    "Total reviews:",
    f"{len(reviews):,}"
)

print("\nReviews by source:")

print(
    reviews["source"]
    .value_counts()
)

print("\nLanguages:")

print(
    reviews["language"]
    .value_counts()
)

display(
    reviews.head()
)

BUILD UNIFIED REVIEW TABLE
Total reviews: 767,179

Reviews by source:
source
amazon    736093
shopee     31086
Name: count, dtype: int64

Languages:
language
en    736093
vi     31086
Name: count, dtype: int64


,review_id,user_id,product_id,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,language,source
0,review_0000000,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B07N69T6TM,1.0,Didn’t work at all lenses loose/broken.,"These didn’t work. Idk if they were damaged in shipping or what, but the lenses were loose or so...",1592678549731,True,0.0,en,amazon
1,review_0000001,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B01G8JO5F2,5.0,Excellent!,I love these. They even come with a carry case and several sizes of ear bud inserts. Thank heave...,1523093017534,True,0.0,en,amazon
2,review_0000002,amazon_AG2L7H23R5LLKDKLBEF2Q3L2MVDA,amazon_B07CJYMRWM,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times because I love them so much and overuse them and find w...,1676601581238,True,0.0,en,amazon
3,review_0000003,amazon_AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,amazon_B07CML419K,5.0,Great Fan! I’m a FAN!,"Light weight, quiet and totally awesome!!! It doesn’t come with a charging base, but the cord en...",1637522881041,True,0.0,en,amazon
4,review_0000004,amazon_AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,amazon_B09S6Y5BRG,5.0,Love the headphones - great range,These are fantastic headphones and I love that i can have my cell phone plugged in at a central ...,1541356831659,True,9.0,en,amazon


# Tạo interactions.csv

In [30]:
# ============================================================
# STEP 1 - CELL 18
# BUILD INTERACTION TABLE
# ============================================================

print("=" * 70)
print("BUILD INTERACTION TABLE")
print("=" * 70)

interactions = (
    reviews[
        [
            "user_id",
            "product_id",
            "rating",
            "timestamp",
            "source"
        ]
    ]
    .copy()
)

print(
    "Interactions:",
    f"{len(interactions):,}"
)

print(
    "Users:",
    f"{interactions['user_id'].nunique():,}"
)

print(
    "Products:",
    f"{interactions['product_id'].nunique():,}"
)

print("\nBy source:")

print(
    interactions["source"]
    .value_counts()
)

display(
    interactions.head()
)

BUILD INTERACTION TABLE
Interactions: 767,179
Users: 228,315
Products: 123,351

By source:
source
amazon    736093
shopee     31086
Name: count, dtype: int64


,user_id,product_id,rating,timestamp,source
0,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B07N69T6TM,1.0,1592678549731,amazon
1,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B01G8JO5F2,5.0,1523093017534,amazon
2,amazon_AG2L7H23R5LLKDKLBEF2Q3L2MVDA,amazon_B07CJYMRWM,5.0,1676601581238,amazon
3,amazon_AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,amazon_B07CML419K,5.0,1637522881041,amazon
4,amazon_AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,amazon_B09S6Y5BRG,5.0,1541356831659,amazon


# Consistency Audit

In [31]:
# ============================================================
# STEP 1 - CELL 19
# FINAL CONSISTENCY AUDIT
# ============================================================

print("=" * 70)
print("FINAL STEP 1 CONSISTENCY AUDIT")
print("=" * 70)

product_ids = set(
    products["product_id"]
)

review_product_ids = set(
    reviews["product_id"]
)

interaction_product_ids = set(
    interactions["product_id"]
)


checks = {
    "products_unique":
        products["product_id"].is_unique,

    "reviews_have_valid_products":
        review_product_ids.issubset(product_ids),

    "interactions_have_valid_products":
        interaction_product_ids.issubset(product_ids),

    "review_ratings_valid":
        reviews["rating"]
        .between(1, 5)
        .all(),

    "interaction_ratings_valid":
        interactions["rating"]
        .between(1, 5)
        .all(),

    "review_user_not_null":
        reviews["user_id"]
        .notna()
        .all(),

    "review_product_not_null":
        reviews["product_id"]
        .notna()
        .all()
}


for name, status in checks.items():

    icon = "✅" if status else "❌"

    print(
        f"{icon} {name:<35}: {status}"
    )


print("\n" + "-" * 70)

print(
    "Products     :",
    f"{len(products):,}"
)

print(
    "Reviews      :",
    f"{len(reviews):,}"
)

print(
    "Interactions :",
    f"{len(interactions):,}"
)

print(
    "Users        :",
    f"{interactions['user_id'].nunique():,}"
)

print("\nProducts by source:")

print(
    products["source"]
    .value_counts()
)

print("\nReviews by source:")

print(
    reviews["source"]
    .value_counts()
)

print("=" * 70)

FINAL STEP 1 CONSISTENCY AUDIT
✅ products_unique                    : True
✅ reviews_have_valid_products        : True
✅ interactions_have_valid_products   : True
✅ review_ratings_valid               : True
✅ interaction_ratings_valid          : True
✅ review_user_not_null               : True
✅ review_product_not_null            : True

----------------------------------------------------------------------
Products     : 164,926
Reviews      : 767,179
Interactions : 767,179
Users        : 228,315

Products by source:
source
amazon    121917
tiki       41575
shopee      1434
Name: count, dtype: int64

Reviews by source:
source
amazon    736093
shopee     31086
Name: count, dtype: int64


# lƯU 

In [32]:
# ============================================================
# STEP 1 - CELL 20
# SAVE FINAL STEP 1 DATASETS
# ============================================================

assert all(checks.values()), \
    "Consistency audit failed. Do not save."

products_path = (
    OUTPUT_DIR /
    "products.csv"
)

reviews_path = (
    OUTPUT_DIR /
    "reviews.csv"
)

interactions_path = (
    OUTPUT_DIR /
    "interactions.csv"
)


products.to_csv(
    products_path,
    index=False
)

reviews.to_csv(
    reviews_path,
    index=False
)

interactions.to_csv(
    interactions_path,
    index=False
)


print("=" * 70)
print("STEP 1 OUTPUT SAVED")
print("=" * 70)

for path in [
    products_path,
    reviews_path,
    interactions_path
]:

    size_mb = (
        path.stat().st_size /
        (1024 ** 2)
    )

    print(
        f"{path.name:<25}"
        f"{size_mb:>10.2f} MB"
    )

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\n✅ STEP 1 COMPLETED")

STEP 1 OUTPUT SAVED
products.csv                  40.26 MB
reviews.csv                  289.88 MB
interactions.csv              57.43 MB

Output directory:
/kaggle/working/step1

✅ STEP 1 COMPLETED
